# Cellpose-SAM 1050-nm red sessions — WSL v2

WSL notebook for local CellposeSAM segmentation on the RTX 3050.

Input:
`/mnt/d/_data/_newAAV_2026/molecular_tracking_derivatives/<mouse>/sessions/<YYYYMMDD>/1050/preprocessing/red.tif`

Output:
`/mnt/d/_data/_newAAV_2026/molecular_tracking_derivatives/<mouse>/sessions/<YYYYMMDD>/1050/segmentation/mask.tif`

Behavior preserved from the established workflow:
- `cpsam_v2`
- `do_3D=True`
- `z_axis=0`
- `channel_axis=3`
- `min_size=100`
- existing `mask.tif` files are skipped unless `OVERWRITE_EXISTING=True`

This v2 also:
- logs runtime per newly segmented session
- logs peak CUDA allocated/reserved memory
- releases large per-session objects after every stack
- runs `gc.collect()` and `torch.cuda.empty_cache()` between stacks


In [1]:
from pathlib import Path
import csv
import gc
import os
import time
import uuid

import numpy as np
import tifffile
import torch
from tqdm.auto import tqdm
from cellpose import models
from cellpose.io import imread_3D


/mnt/d/Codex_folder/molecular_tracking/.venv-cellpose/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DERIVATIVES_ROOT = Path("/mnt/d/_data/_newAAV_2026/molecular_tracking_derivatives")

# None = discover all mice with matching data.
MOUSE_IDS = ['Fucci-Tri_5', 'Fucci-Tri_6', 'Fucci-Tri_7',
             'Fucci-Dead_3', 'Fucci-Dead_4',
             'Fucci-Dead_1', 'Fucci-Dead_2', 'Fucci-Tri_3', 'Fucci-Tri_4']


OVERWRITE_EXISTING = False
MIN_SIZE = 100

if not DERIVATIVES_ROOT.is_dir():
    raise FileNotFoundError(f"Derivatives root not found: {DERIVATIVES_ROOT}")


In [3]:
# Discover canonical 1050-nm red stacks.
image_paths = []

mouse_dirs = sorted(
    p for p in DERIVATIVES_ROOT.iterdir()
    if p.is_dir() and not p.name.startswith("_")
)

if MOUSE_IDS is not None:
    wanted = set(MOUSE_IDS)
    mouse_dirs = [p for p in mouse_dirs if p.name in wanted]

for mouse_dir in mouse_dirs:
    sessions_dir = mouse_dir / "sessions"
    if not sessions_dir.is_dir():
        continue

    for session_dir in sorted(p for p in sessions_dir.iterdir() if p.is_dir()):
        image_path = session_dir / "1050" / "preprocessing" / "red.tif"
        if image_path.is_file():
            image_paths.append(image_path)

if not image_paths:
    raise FileNotFoundError("No matching 1050-nm red stacks were found.")

print(f"Found {len(image_paths)} stack(s).")
for p in image_paths:
    print(p)


Found 107 stack(s).
/mnt/d/_data/_newAAV_2026/molecular_tracking_derivatives/Fucci-Dead_1/sessions/20260819/1050/preprocessing/red.tif
/mnt/d/_data/_newAAV_2026/molecular_tracking_derivatives/Fucci-Dead_1/sessions/20260820/1050/preprocessing/red.tif
/mnt/d/_data/_newAAV_2026/molecular_tracking_derivatives/Fucci-Dead_1/sessions/20260821/1050/preprocessing/red.tif
/mnt/d/_data/_newAAV_2026/molecular_tracking_derivatives/Fucci-Dead_1/sessions/20260824/1050/preprocessing/red.tif
/mnt/d/_data/_newAAV_2026/molecular_tracking_derivatives/Fucci-Dead_1/sessions/20260825/1050/preprocessing/red.tif
/mnt/d/_data/_newAAV_2026/molecular_tracking_derivatives/Fucci-Dead_1/sessions/20260827/1050/preprocessing/red.tif
/mnt/d/_data/_newAAV_2026/molecular_tracking_derivatives/Fucci-Dead_1/sessions/20260828/1050/preprocessing/red.tif
/mnt/d/_data/_newAAV_2026/molecular_tracking_derivatives/Fucci-Dead_1/sessions/20260831/1050/preprocessing/red.tif
/mnt/d/_data/_newAAV_2026/molecular_tracking_derivatives/Fuc

In [4]:
model = models.CellposeModel(
    gpu=True,
    pretrained_model="cpsam_v2",
)

print("Model device:", model.device)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


Model device: cuda:0
GPU: NVIDIA GeForce RTX 3050


In [5]:
results = []

for image_path in tqdm(image_paths, desc="Cellpose-SAM 1050 sessions"):
    mouse_id = image_path.parents[4].name
    session_id = image_path.parents[2].name
    segmentation_dir = image_path.parent.parent / "segmentation"
    output_path = segmentation_dir / "mask.tif"

    if output_path.exists() and not OVERWRITE_EXISTING:
        tqdm.write(f"Skipping existing: {mouse_id} {session_id} -> {output_path}")
        results.append({
            "mouse_id": mouse_id,
            "session_id": session_id,
            "red_path": str(image_path),
            "mask_path": str(output_path),
            "status": "ALREADY_EXISTS",
            "n_masks": "",
            "runtime_min": "",
            "peak_allocated_gb": "",
            "peak_reserved_gb": "",
            "error": "",
        })
        continue

    loaded_image = masks = flows = styles = mask_to_save = check = None
    temp_path = None

    try:
        tqdm.write(f"Processing: {mouse_id} {session_id}")
        loaded_image = imread_3D(image_path.as_posix())

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()

        t0 = time.perf_counter()

        masks, flows, styles = model.eval(
            loaded_image,
            do_3D=True,
            z_axis=0,
            channel_axis=3,
            min_size=MIN_SIZE,
        )

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        runtime_min = (time.perf_counter() - t0) / 60.0
        n_masks = int(np.max(masks))

        if torch.cuda.is_available():
            peak_allocated_gb = torch.cuda.max_memory_allocated() / 1024**3
            peak_reserved_gb = torch.cuda.max_memory_reserved() / 1024**3
        else:
            peak_allocated_gb = float("nan")
            peak_reserved_gb = float("nan")

        tqdm.write(
            f"  Found {n_masks} mask(s) | Runtime: {runtime_min:.2f} min | "
            f"Peak GPU: {peak_allocated_gb:.2f} GB allocated, "
            f"{peak_reserved_gb:.2f} GB reserved"
        )

        segmentation_dir.mkdir(parents=True, exist_ok=True)

        dtype = np.uint16 if n_masks <= np.iinfo(np.uint16).max else np.uint32
        mask_to_save = masks.astype(dtype, copy=False)

        temp_path = segmentation_dir / f".mask.tif.tmp.{uuid.uuid4().hex}"

        tifffile.imwrite(
            temp_path,
            mask_to_save,
            photometric="minisblack",
            metadata={"axes": "ZYX"},
        )

        check = tifffile.imread(temp_path)

        if check.shape != mask_to_save.shape:
            raise RuntimeError(
                f"Saved mask shape mismatch: expected {mask_to_save.shape}, got {check.shape}"
            )

        if int(np.max(check)) != n_masks:
            raise RuntimeError(
                f"Saved mask label check failed: expected {n_masks}, got {int(np.max(check))}"
            )

        if output_path.exists():
            raise FileExistsError(
                f"Destination appeared during processing; refusing to overwrite: {output_path}"
            )

        os.rename(temp_path, output_path)
        temp_path = None

        results.append({
            "mouse_id": mouse_id,
            "session_id": session_id,
            "red_path": str(image_path),
            "mask_path": str(output_path),
            "status": "SEGMENTED",
            "n_masks": n_masks,
            "runtime_min": round(runtime_min, 3),
            "peak_allocated_gb": round(peak_allocated_gb, 3),
            "peak_reserved_gb": round(peak_reserved_gb, 3),
            "error": "",
        })

        tqdm.write(f"Saved: {output_path}")

    except Exception as error:
        if temp_path is not None and temp_path.exists():
            temp_path.unlink()

        results.append({
            "mouse_id": mouse_id,
            "session_id": session_id,
            "red_path": str(image_path),
            "mask_path": str(output_path),
            "status": "FAILED",
            "n_masks": "",
            "runtime_min": "",
            "peak_allocated_gb": "",
            "peak_reserved_gb": "",
            "error": str(error),
        })

        tqdm.write(f"FAILED: {mouse_id} {session_id}")
        tqdm.write(f"  {error}")

    finally:
        # Release large per-session objects but keep the Cellpose model loaded.
        loaded_image = None
        masks = None
        flows = None
        styles = None
        mask_to_save = None
        check = None

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            allocated_gb = torch.cuda.memory_allocated() / 1024**3
            reserved_gb = torch.cuda.memory_reserved() / 1024**3
            tqdm.write(
                f"  GPU after cleanup: {allocated_gb:.2f} GB allocated, "
                f"{reserved_gb:.2f} GB reserved"
            )


Cellpose-SAM 1050 sessions:   0%|          | 0/107 [00:00<?, ?it/s]

Skipping existing: Fucci-Dead_1 20260819 -> /mnt/d/_data/_newAAV_2026/molecular_tracking_derivatives/Fucci-Dead_1/sessions/20260819/1050/segmentation/mask.tif
Skipping existing: Fucci-Dead_1 20260820 -> /mnt/d/_data/_newAAV_2026/molecular_tracking_derivatives/Fucci-Dead_1/sessions/20260820/1050/segmentation/mask.tif
Skipping existing: Fucci-Dead_1 20260821 -> /mnt/d/_data/_newAAV_2026/molecular_tracking_derivatives/Fucci-Dead_1/sessions/20260821/1050/segmentation/mask.tif
Skipping existing: Fucci-Dead_1 20260824 -> /mnt/d/_data/_newAAV_2026/molecular_tracking_derivatives/Fucci-Dead_1/sessions/20260824/1050/segmentation/mask.tif
Skipping existing: Fucci-Dead_1 20260825 -> /mnt/d/_data/_newAAV_2026/molecular_tracking_derivatives/Fucci-Dead_1/sessions/20260825/1050/segmentation/mask.tif
Skipping existing: Fucci-Dead_1 20260827 -> /mnt/d/_data/_newAAV_2026/molecular_tracking_derivatives/Fucci-Dead_1/sessions/20260827/1050/segmentation/mask.tif
Skipping existing: Fucci-Dead_1 20260828 -> /m

Cellpose-SAM 1050 sessions:  23%|██▎       | 25/107 [00:00<00:00, 244.22it/s]

Skipping existing: Fucci-Dead_1 20260908 -> /mnt/d/_data/_newAAV_2026/molecular_tracking_derivatives/Fucci-Dead_1/sessions/20260908/1050/segmentation/mask.tif
Skipping existing: Fucci-Dead_1 20260909 -> /mnt/d/_data/_newAAV_2026/molecular_tracking_derivatives/Fucci-Dead_1/sessions/20260909/1050/segmentation/mask.tif
Skipping existing: Fucci-Dead_1 20260910 -> /mnt/d/_data/_newAAV_2026/molecular_tracking_derivatives/Fucci-Dead_1/sessions/20260910/1050/segmentation/mask.tif
Skipping existing: Fucci-Dead_1 20260911 -> /mnt/d/_data/_newAAV_2026/molecular_tracking_derivatives/Fucci-Dead_1/sessions/20260911/1050/segmentation/mask.tif
Skipping existing: Fucci-Dead_1 20260914 -> /mnt/d/_data/_newAAV_2026/molecular_tracking_derivatives/Fucci-Dead_1/sessions/20260914/1050/segmentation/mask.tif
Skipping existing: Fucci-Dead_1 20260915 -> /mnt/d/_data/_newAAV_2026/molecular_tracking_derivatives/Fucci-Dead_1/sessions/20260915/1050/segmentation/mask.tif
Skipping existing: Fucci-Dead_1 20260916 -> /m

Cellpose-SAM 1050 sessions:  47%|████▋     | 50/107 [00:00<00:00, 240.32it/s]

...

  Found 6298 mask(s) | Runtime: 21.38 min | Peak GPU: 1.84 GB allocated, 2.69 GB reserved


Cellpose-SAM 1050 sessions:  51%|█████▏    | 55/107 [46:26<1:19:18, 91.51s/it]

Saved: /mnt/d/_data/_newAAV_2026/molecular_tracking_derivatives/Fucci-Dead_4/sessions/20261007/1050/segmentation/mask.tif
  GPU after cleanup: 0.58 GB allocated, 0.61 GB reserved
Skipping existing: Fucci-Tri_3 20260904 -> /mnt/d/_data/_newAAV_2026/molecular_tracking_derivatives/Fucci-Tri_3/sessions/20260904/1050/segmentation/mask.tif
Skipping existing: Fucci-Tri_3 20260908 -> /mnt/d/_data/_newAAV_2026/molecular_tracking_derivatives/Fucci-Tri_3/sessions/20260908/1050/segmentation/mask.tif
Skipping existing: Fucci-Tri_3 20260909 -> /mnt/d/_data/_newAAV_2026/molecular_tracking_derivatives/Fucci-Tri_3/sessions/20260909/1050/segmentation/mask.tif
Skipping existing: Fucci-Tri_3 20260910 -> /mnt/d/_data/_newAAV_2026/molecular_tracking_derivatives/Fucci-Tri_3/sessions/20260910/1050/segmentation/mask.tif
Skipping existing: Fucci-Tri_3 20260911 -> /mnt/d/_data/_newAAV_2026/molecular_tracking_derivatives/Fucci-Tri_3/sessions/20260911/1050/segmentation/mask.tif
Skipping existing: Fucci-Tri_3 20260

Cellpose-SAM 1050 sessions: 100%|██████████| 107/107 [46:26<00:00, 26.05s/it] 


Skipping existing: Fucci-Tri_3 20260921 -> /mnt/d/_data/_newAAV_2026/molecular_tracking_derivatives/Fucci-Tri_3/sessions/20260921/1050/segmentation/mask.tif
Skipping existing: Fucci-Tri_3 20260923 -> /mnt/d/_data/_newAAV_2026/molecular_tracking_derivatives/Fucci-Tri_3/sessions/20260923/1050/segmentation/mask.tif
Skipping existing: Fucci-Tri_3 20260924 -> /mnt/d/_data/_newAAV_2026/molecular_tracking_derivatives/Fucci-Tri_3/sessions/20260924/1050/segmentation/mask.tif
Skipping existing: Fucci-Tri_3 20260925 -> /mnt/d/_data/_newAAV_2026/molecular_tracking_derivatives/Fucci-Tri_3/sessions/20260925/1050/segmentation/mask.tif
Skipping existing: Fucci-Tri_3 20260928 -> /mnt/d/_data/_newAAV_2026/molecular_tracking_derivatives/Fucci-Tri_3/sessions/20260928/1050/segmentation/mask.tif
Skipping existing: Fucci-Tri_3 20260929 -> /mnt/d/_data/_newAAV_2026/molecular_tracking_derivatives/Fucci-Tri_3/sessions/20260929/1050/segmentation/mask.tif
Skipping existing: Fucci-Tri_3 20260930 -> /mnt/d/_data/_n

In [6]:
log_path = DERIVATIVES_ROOT / "cellposeSAM_1050_session_log.csv"

fieldnames = [
    "mouse_id",
    "session_id",
    "red_path",
    "mask_path",
    "status",
    "n_masks",
    "runtime_min",
    "peak_allocated_gb",
    "peak_reserved_gb",
    "error",
]

with log_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results)

print("\nFinished.")
print("Log:", log_path)

status_counts = {}
for row in results:
    status_counts[row["status"]] = status_counts.get(row["status"], 0) + 1

for status, count in sorted(status_counts.items()):
    print(f"{status:20s} {count:4d}")



Finished.
Log: /mnt/d/_data/_newAAV_2026/molecular_tracking_derivatives/cellposeSAM_1050_session_log.csv
ALREADY_EXISTS        105
SEGMENTED               2
